In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score
from pathlib import Path

CSV = Path(r"C:\Users\Vivek\Documents\dissertation\evaluation\spike_v4_scores.csv")
df = pd.read_csv(CSV)

axes = [
    "text_legibility",
    "regional_appropriateness",
    "packaging_plausibility",
    "visual_quality",
]

print(f"Total scored rows: {len(df)}")
print()
print(f"{'Axis':<30} {'Kappa':>8} {'Mean S1':>8} {'Mean S2':>8} {'Δ Mean':>8} {'% Exact':>8} {'% ±1':>8}")
print("-" * 88)

for axis in axes:
    s1 = pd.to_numeric(df[f"session1_{axis}"], errors="coerce")
    s2 = pd.to_numeric(df[f"session2_{axis}"], errors="coerce")
    mask = s1.notna() & s2.notna()
    s1v, s2v = s1[mask].astype(int), s2[mask].astype(int)
    kappa = cohen_kappa_score(s1v, s2v, weights="linear")
    diff = np.abs(s1v - s2v)
    print(f"{axis:<30} {kappa:>8.3f} {s1v.mean():>8.2f} {s2v.mean():>8.2f} "
          f"{s2v.mean()-s1v.mean():>+8.2f} {(diff==0).mean()*100:>7.1f}% {(diff<=1).mean()*100:>7.1f}%")

print()
print("Per-image total disagreement:")
total_diff = sum(
    np.abs(
        pd.to_numeric(df[f"session1_{ax}"], errors="coerce").fillna(0).astype(int) -
        pd.to_numeric(df[f"session2_{ax}"], errors="coerce").fillna(0).astype(int)
    )
    for ax in axes
)
print(f"  Mean: {total_diff.mean():.2f}  |  Max: {total_diff.max()}  |  Zero-disagreement rows: {(total_diff==0).sum()} ({(total_diff==0).mean()*100:.1f}%)")

Total scored rows: 12

Axis                              Kappa  Mean S1  Mean S2   Δ Mean  % Exact     % ±1
----------------------------------------------------------------------------------------
text_legibility                     nan     1.00     1.00    +0.00   100.0%   100.0%
regional_appropriateness          0.647     1.33     1.67    +0.33    66.7%   100.0%
packaging_plausibility            0.063     2.17     2.42    +0.25    58.3%   100.0%
visual_quality                    0.789     1.25     1.42    +0.17    83.3%   100.0%

Per-image total disagreement:
  Mean: 0.92  |  Max: 2  |  Zero-disagreement rows: 3 (25.0%)


C:\Users\Vivek\Documents\dissertation\sdxl-env\Lib\site-packages\sklearn\metrics\_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\Vivek\Documents\dissertation\sdxl-env\Lib\site-packages\sklearn\utils\_param_validation.py:218: UndefinedMetricWarning: `y1`, `y2` and `labels` have only one label in common. `cohen_kappa_score` is undefined and set to the value defined by the the `replace_undefined_by` param, which is set to nan.
  return func(*args, **kwargs)
